# Validação


In [1]:
%%html
<link rel="stylesheet" href="./style.css">


In [2]:
from balancing import compare_smote
from helper_for_validation import (
    display_cross_validation_results,
    display_cross_validation_summary,
    display_grid_search_results,
    display_grid_search_summary,
    display_train_test_target_distribution,
    get_best_grid_search_parameters,
)
from pipelines import (
    _build_decision_tree_pipeline,
    _build_random_forest_pipeline,
    input_data_for_train,
    seeds,
    target_data_for_test,
    target_data_for_train,
)
from tuning import (
    run_grid_searches_for_decision_tree,
    run_grid_searches_for_random_forest,
)

## Separação dos dados

Separamos os dados em **treino** (`80%`) e **teste** (`20%`) de forma estratificada pelo atributo objetivo:

- **Age group**: `10-19`, `20-29`, `30-39`, `40-49`, `50-59`, `60-69`, `70+`.


In [3]:
display_train_test_target_distribution(
    target_data_for_train,
    target_data_for_test,
);

Age Group,Train N,Test N,Total N,Train %,Test %
10-19,46,12,58,79.3%,20.7%
20-29,158,40,198,79.8%,20.2%
30-39,90,22,112,80.4%,19.6%
40-49,122,31,153,79.7%,20.3%
50-59,162,40,202,80.2%,19.8%
60-69,92,23,115,80.0%,20.0%
70+,69,17,86,80.2%,19.8%
General,739,185,924,80.0%,20.0%


## Balanceamento


### Decision tree


In [4]:
smote_comparison_for_decision_tree, smote_results_for_decision_tree = compare_smote(
    build_pipeline=_build_decision_tree_pipeline,
    input_data=input_data_for_train,
    target_data=target_data_for_train,
    seeds=seeds,
)

In [5]:
metric_columns = [
    "Accuracy",
    "Balanced Accuracy",
    "Macro F1",
]

display_cross_validation_results(
    smote_results_for_decision_tree,
    metric_columns=metric_columns,
    caption="Decision Tree: SMOTE Cross-Validation Results",
)

display_cross_validation_summary(
    smote_comparison_for_decision_tree,
    metric_columns=metric_columns,
    caption="Decision Tree: SMOTE Comparison",
)

Seed,Strategy,Accuracy,Balanced Accuracy,Macro F1
27,Without SMOTE,0.355,0.342,0.332
27,With SMOTE,0.346,0.361,0.336
32,Without SMOTE,0.382,0.379,0.358
32,With SMOTE,0.382,0.402,0.368
59,Without SMOTE,0.332,0.333,0.320
59,With SMOTE,0.361,0.385,0.358
74,Without SMOTE,0.372,0.371,0.347
74,With SMOTE,0.380,0.394,0.365
93,Without SMOTE,0.352,0.357,0.339
93,With SMOTE,0.368,0.385,0.352


Strategy,Accuracy,Balanced Accuracy,Macro F1
With SMOTE,0.368 ± 0.015,0.385 ± 0.015,0.356 ± 0.013
Without SMOTE,0.358 ± 0.019,0.356 ± 0.019,0.339 ± 0.014


### Random forest


In [6]:
smote_comparison_for_random_forest, smote_results_for_random_forest = compare_smote(
    build_pipeline=_build_random_forest_pipeline,
    input_data=input_data_for_train,
    target_data=target_data_for_train,
    seeds=seeds,
)

In [7]:
display_cross_validation_results(
    smote_results_for_random_forest,
    metric_columns=metric_columns,
    caption="Random Forest: SMOTE Cross-Validation Results",
)

display_cross_validation_summary(
    smote_comparison_for_random_forest,
    metric_columns=metric_columns,
    caption="Random Forest: SMOTE Comparison",
)

Seed,Strategy,Accuracy,Balanced Accuracy,Macro F1
27,Without SMOTE,0.384,0.364,0.362
27,With SMOTE,0.363,0.371,0.352
32,Without SMOTE,0.410,0.388,0.384
32,With SMOTE,0.409,0.410,0.394
59,Without SMOTE,0.383,0.365,0.364
59,With SMOTE,0.394,0.407,0.388
74,Without SMOTE,0.396,0.384,0.372
74,With SMOTE,0.387,0.394,0.373
93,Without SMOTE,0.417,0.403,0.399
93,With SMOTE,0.394,0.401,0.375


Strategy,Accuracy,Balanced Accuracy,Macro F1
With SMOTE,0.389 ± 0.017,0.396 ± 0.016,0.376 ± 0.016
Without SMOTE,0.398 ± 0.015,0.381 ± 0.017,0.376 ± 0.015


## Ajuste de hiper-parâmetros


### Decision tree


Utilizamos o **GridSearch** para encontrarmos os melhores valores de hiper-parâmetros.

- Testamos os seguintes **critérios** de decisão: [`gini`, `entropy`].
- Testamos as seguintes **profundidades** máximas: [`7`, `10`, `15`, `None`].
- Testamos as seguintes quantidades mínimas de **amostras** para divisão de nó: [`2`, `5`, `10`].
- Testamos as seguintes quantidades mínimas de **amostras** que um nó folha deve ter: [`1`, `2`, `5`].

Como **critério de seleção**, utilizamos `f1_macro`.

- A métrica `F1` que combina _precision_ e _recall_ para cada classe objetivo.
- O `F1 macro` calcula a média dos valores de F1 das três.

Executamos o GridSearch em `5` **folds** estratificados no conjunto de `treino`.


In [8]:
param_grid_of_decision_tree = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [7, 10, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 5],
}

#### Without SMOTE


In [9]:
grid_searches_for_decision_tree_without_smote = run_grid_searches_for_decision_tree(
    input_data_for_train,
    target_data_for_train,
    seeds=seeds,
    param_grid=param_grid_of_decision_tree,
    use_smote=False,
)

In [10]:
best_parameters_for_decision_tree_without_smote = get_best_grid_search_parameters(
    grid_searches_for_decision_tree_without_smote
)
display(best_parameters_for_decision_tree_without_smote)

{'classifier__criterion': 'entropy',
 'classifier__max_depth': None,
 'classifier__min_samples_leaf': 1,
 'classifier__min_samples_split': 10}

In [11]:
display_grid_search_results(
    grid_searches_for_decision_tree_without_smote,
    caption="Decision Tree without SMOTE: Grid Search Results",
)

Seed,Criterion,Max Depth,Min Samples Leaf,Min Samples Split,F1 Macro,F1 Macro Std,Balanced Accuracy,Balanced Accuracy Std,Accuracy,Accuracy Std
27,entropy,None,1,10,0.3659,0.0337,0.3757,0.0355,0.3924,0.0277
27,entropy,None,2,10,0.3659,0.0331,0.3748,0.0350,0.3924,0.0286
27,entropy,None,1,5,0.3641,0.0169,0.3765,0.0140,0.3829,0.0028
27,entropy,7,1,2,0.3591,0.0277,0.3698,0.0317,0.3965,0.0190
27,entropy,10,1,10,0.3585,0.0283,0.3690,0.0249,0.3843,0.0228
27,entropy,10,2,10,0.3553,0.0310,0.3644,0.0264,0.3816,0.0301
27,entropy,10,1,2,0.3550,0.0135,0.3669,0.0072,0.3870,0.0214
27,entropy,7,1,10,0.3507,0.0330,0.3663,0.0355,0.3911,0.0231
27,entropy,10,1,5,0.3490,0.0207,0.3619,0.0213,0.3748,0.0147
27,entropy,7,2,10,0.3481,0.0237,0.3638,0.0260,0.3897,0.0201


In [12]:
display_grid_search_summary(
    grid_searches_for_decision_tree_without_smote,
    caption="Decision Tree without SMOTE: Grid Search Comparison",
)

Criterion,Max Depth,Min Samples Split,Min Samples Leaf,F1 Macro,Balanced Accuracy,Accuracy
entropy,None,10,1,0.3515 ± 0.0113,0.3645 ± 0.0096,0.3800 ± 0.0093
entropy,None,10,2,0.3495 ± 0.0095,0.3615 ± 0.0078,0.3811 ± 0.0073
entropy,10,10,2,0.3491 ± 0.0056,0.3596 ± 0.0088,0.3822 ± 0.0062
entropy,10,2,1,0.3488 ± 0.0085,0.3627 ± 0.0070,0.3743 ± 0.0127
entropy,10,10,1,0.3476 ± 0.0068,0.3588 ± 0.0069,0.3784 ± 0.0049
entropy,None,5,1,0.3473 ± 0.0140,0.3673 ± 0.0134,0.3700 ± 0.0110
entropy,10,5,1,0.3454 ± 0.0028,0.3618 ± 0.0054,0.3735 ± 0.0058
entropy,None,2,1,0.3443 ± 0.0091,0.3620 ± 0.0110,0.3616 ± 0.0097
gini,None,10,1,0.3431 ± 0.0083,0.3577 ± 0.0123,0.3740 ± 0.0090
entropy,7,2,1,0.3427 ± 0.0114,0.3558 ± 0.0115,0.3873 ± 0.0080


#### With SMOTE


In [13]:
grid_searches_for_decision_tree_with_smote = run_grid_searches_for_decision_tree(
    input_data_for_train,
    target_data_for_train,
    seeds=seeds,
    param_grid=param_grid_of_decision_tree,
    use_smote=True,
)

In [14]:
best_parameters_for_decision_tree_with_smote = get_best_grid_search_parameters(
    grid_searches_for_decision_tree_with_smote
)
display(best_parameters_for_decision_tree_with_smote)

{'classifier__criterion': 'entropy',
 'classifier__max_depth': 7,
 'classifier__min_samples_leaf': 1,
 'classifier__min_samples_split': 5}

In [15]:
display_grid_search_results(
    grid_searches_for_decision_tree_with_smote,
    caption="Decision Tree with SMOTE: Grid Search Results",
)

Seed,Criterion,Max Depth,Min Samples Leaf,Min Samples Split,F1 Macro,F1 Macro Std,Balanced Accuracy,Balanced Accuracy Std,Accuracy,Accuracy Std
27,entropy,None,2,10,0.3619,0.0227,0.3837,0.0279,0.3857,0.0224
27,entropy,None,5,2,0.3599,0.0243,0.3814,0.0276,0.3816,0.0265
27,entropy,None,5,5,0.3599,0.0243,0.3814,0.0276,0.3816,0.0265
27,entropy,None,5,10,0.3599,0.0243,0.3814,0.0276,0.3816,0.0265
27,entropy,7,1,2,0.3598,0.0666,0.3929,0.0675,0.3924,0.0522
27,entropy,7,1,5,0.3572,0.0579,0.3918,0.0623,0.3910,0.0469
27,gini,7,1,5,0.3562,0.0337,0.3880,0.0476,0.3775,0.0265
27,entropy,None,1,10,0.3547,0.0184,0.3787,0.0333,0.3789,0.0231
27,gini,7,1,10,0.3546,0.0350,0.3895,0.0529,0.3775,0.0239
27,entropy,7,2,2,0.3533,0.0551,0.3898,0.0607,0.3870,0.0445


In [16]:
display_grid_search_summary(
    grid_searches_for_decision_tree_with_smote,
    caption="Decision Tree with SMOTE: Grid Search Comparison",
)

Criterion,Max Depth,Min Samples Split,Min Samples Leaf,F1 Macro,Balanced Accuracy,Accuracy
entropy,7,5,1,0.3625 ± 0.0159,0.3978 ± 0.0131,0.3899 ± 0.0099
entropy,7,2,1,0.3620 ± 0.0176,0.3976 ± 0.0146,0.3889 ± 0.0109
entropy,7,2,5,0.3589 ± 0.0157,0.3909 ± 0.0158,0.3897 ± 0.0159
entropy,7,5,5,0.3589 ± 0.0157,0.3909 ± 0.0158,0.3897 ± 0.0159
entropy,7,10,5,0.3589 ± 0.0157,0.3909 ± 0.0158,0.3897 ± 0.0159
entropy,7,10,1,0.3582 ± 0.0112,0.3952 ± 0.0121,0.3883 ± 0.0078
entropy,7,5,2,0.3580 ± 0.0179,0.3951 ± 0.0151,0.3859 ± 0.0119
entropy,7,2,2,0.3577 ± 0.0193,0.3948 ± 0.0156,0.3870 ± 0.0132
gini,None,10,1,0.3567 ± 0.0111,0.3817 ± 0.0129,0.3743 ± 0.0105
entropy,7,10,2,0.3560 ± 0.0111,0.3932 ± 0.0115,0.3861 ± 0.0088


### Random forest


Utilizamos o **GridSearch** para encontrarmos os melhores valores de hiper-parâmetros.

- Testamos os seguintes **critérios** de decisão: [`gini`, `entropy`].
- Testamos as seguintes **profundidades** máximas: [`5`, `10`, `None`].
- Testamos as seguintes quantidades mínimas de **amostras** para divisão de nó: [`2`, `5`].
- Testamos as seguintes quantidades mínimas de **amostras** que um nó folha deve ter: [`1`, `2`].
- Testamos as seguintes quantidades de **árvores** na floresta: [`100`, `200`, `300`].
- Testamos os seguintes critérios de **corte**: [`sqrt`, `log2`].

Como **critério de seleção**, utilizamos `f1_macro`.

- A métrica `F1` que combina _precision_ e _recall_ para cada classe objetivo.
- O `F1 macro` calcula a média dos valores de F1 das três.

Executamos o GridSearch em `5` **folds** estratificados no conjunto de `treino`.


In [17]:
param_grid_of_random_forest = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [5, 10, None],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2],
    "classifier__max_features": ["sqrt", "log2"],
}

#### Without SMOTE


In [18]:
grid_searches_for_random_forest_without_smote = run_grid_searches_for_random_forest(
    input_data_for_train,
    target_data_for_train,
    seeds=seeds,
    param_grid=param_grid_of_random_forest,
    use_smote=False,
)

/home/gabriel/dev/data_mining/assignment_7/.venv/lib64/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/gabriel/dev/data_mining/assignment_7/.venv/lib64/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/gabriel/dev/data_mining/assignment_7/.venv/lib64/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warn

In [19]:
best_parameters_for_random_forest_without_smote = get_best_grid_search_parameters(
    grid_searches_for_random_forest_without_smote
)
display(best_parameters_for_random_forest_without_smote)

{'classifier__criterion': 'entropy',
 'classifier__max_depth': 10,
 'classifier__max_features': 'log2',
 'classifier__min_samples_leaf': 1,
 'classifier__min_samples_split': 5,
 'classifier__n_estimators': 300}

In [20]:
display_grid_search_results(
    grid_searches_for_random_forest_without_smote,
    caption="Random Forest without SMOTE: Grid Search Results",
)

Seed,Criterion,Max Depth,Max Features,Min Samples Leaf,Min Samples Split,N estimators,F1 Macro,F1 Macro Std,Balanced Accuracy,Balanced Accuracy Std,Accuracy,Accuracy Std
27,gini,10,sqrt,1,5,200,0.3962,0.0252,0.3929,0.0230,0.4371,0.0200
27,gini,10,log2,1,5,200,0.3962,0.0252,0.3929,0.0230,0.4371,0.0200
27,gini,10,sqrt,1,5,300,0.3935,0.0278,0.3893,0.0254,0.4317,0.0208
27,gini,10,log2,1,5,300,0.3935,0.0278,0.3893,0.0254,0.4317,0.0208
27,entropy,10,sqrt,2,2,300,0.3899,0.0180,0.3889,0.0202,0.4289,0.0146
27,entropy,10,log2,2,2,300,0.3899,0.0180,0.3889,0.0202,0.4289,0.0146
27,entropy,10,sqrt,2,2,200,0.3894,0.0200,0.3902,0.0211,0.4330,0.0138
27,entropy,10,log2,2,2,200,0.3894,0.0200,0.3902,0.0211,0.4330,0.0138
27,gini,10,sqrt,2,2,300,0.3891,0.0247,0.3884,0.0251,0.4303,0.0169
27,gini,10,log2,2,2,300,0.3891,0.0247,0.3884,0.0251,0.4303,0.0169


In [21]:
display_grid_search_summary(
    grid_searches_for_random_forest_without_smote,
    caption="Random Forest without SMOTE: Grid Search Comparison",
)

Criterion,Max Depth,Min Samples Split,Min Samples Leaf,Max Features,N estimators,F1 Macro,Balanced Accuracy,Accuracy
entropy,10,5,1,log2,300,0.3914 ± 0.0109,0.3914 ± 0.0124,0.4290 ± 0.0114
entropy,10,5,1,sqrt,300,0.3914 ± 0.0109,0.3914 ± 0.0124,0.4290 ± 0.0114
entropy,10,2,2,log2,200,0.3912 ± 0.0079,0.3914 ± 0.0084,0.4295 ± 0.0074
entropy,10,2,2,sqrt,200,0.3912 ± 0.0079,0.3914 ± 0.0084,0.4295 ± 0.0074
gini,10,2,1,log2,300,0.3906 ± 0.0113,0.3935 ± 0.0131,0.4270 ± 0.0089
gini,10,2,1,sqrt,300,0.3906 ± 0.0113,0.3935 ± 0.0131,0.4270 ± 0.0089
entropy,10,2,1,log2,200,0.3904 ± 0.0174,0.3918 ± 0.0167,0.4249 ± 0.0148
entropy,10,2,1,sqrt,200,0.3904 ± 0.0174,0.3918 ± 0.0167,0.4249 ± 0.0148
entropy,10,2,2,log2,300,0.3903 ± 0.0088,0.3922 ± 0.0110,0.4303 ± 0.0094
entropy,10,2,2,sqrt,300,0.3903 ± 0.0088,0.3922 ± 0.0110,0.4303 ± 0.0094


#### With SMOTE


In [22]:
grid_searches_for_random_forest_with_smote = run_grid_searches_for_random_forest(
    input_data_for_train,
    target_data_for_train,
    seeds=seeds,
    param_grid=param_grid_of_random_forest,
    use_smote=True,
)

In [23]:
best_parameters_for_random_forest_with_smote = get_best_grid_search_parameters(
    grid_searches_for_random_forest_with_smote
)
display(best_parameters_for_random_forest_with_smote)

{'classifier__criterion': 'entropy',
 'classifier__max_depth': 10,
 'classifier__max_features': 'log2',
 'classifier__min_samples_leaf': 2,
 'classifier__min_samples_split': 5,
 'classifier__n_estimators': 200}

In [24]:
display_grid_search_results(
    grid_searches_for_random_forest_with_smote,
    caption="Random Forest with SMOTE: Grid Search Results",
)

Seed,Criterion,Max Depth,Max Features,Min Samples Leaf,Min Samples Split,N estimators,F1 Macro,F1 Macro Std,Balanced Accuracy,Balanced Accuracy Std,Accuracy,Accuracy Std
27,gini,10,sqrt,2,5,200,0.4064,0.0102,0.4187,0.0230,0.4236,0.0212
27,gini,10,log2,2,5,200,0.4064,0.0102,0.4187,0.0230,0.4236,0.0212
27,entropy,10,sqrt,2,2,100,0.4012,0.0245,0.4160,0.0374,0.4195,0.0258
27,entropy,10,log2,2,2,100,0.4012,0.0245,0.4160,0.0374,0.4195,0.0258
27,gini,10,sqrt,2,2,200,0.3997,0.0081,0.4153,0.0179,0.4222,0.0200
27,gini,10,log2,2,2,200,0.3997,0.0081,0.4153,0.0179,0.4222,0.0200
27,entropy,10,sqrt,2,5,200,0.3979,0.0108,0.4115,0.0238,0.4168,0.0194
27,entropy,10,log2,2,5,200,0.3979,0.0108,0.4115,0.0238,0.4168,0.0194
27,gini,10,sqrt,2,2,100,0.3978,0.0081,0.4155,0.0192,0.4209,0.0195
27,gini,10,log2,2,2,100,0.3978,0.0081,0.4155,0.0192,0.4209,0.0195


In [25]:
display_grid_search_summary(
    grid_searches_for_random_forest_with_smote,
    caption="Random Forest with SMOTE: Grid Search Comparison",
)

Criterion,Max Depth,Min Samples Split,Min Samples Leaf,Max Features,N estimators,F1 Macro,Balanced Accuracy,Accuracy
entropy,10,5,2,log2,200,0.4039 ± 0.0117,0.4182 ± 0.0114,0.4206 ± 0.0116
entropy,10,5,2,sqrt,200,0.4039 ± 0.0117,0.4182 ± 0.0114,0.4206 ± 0.0116
entropy,10,5,1,log2,300,0.4027 ± 0.0157,0.4187 ± 0.0151,0.4187 ± 0.0150
entropy,10,5,1,sqrt,300,0.4027 ± 0.0157,0.4187 ± 0.0151,0.4187 ± 0.0150
entropy,10,2,2,log2,100,0.4011 ± 0.0049,0.4172 ± 0.0042,0.4181 ± 0.0045
entropy,10,2,2,sqrt,100,0.4011 ± 0.0049,0.4172 ± 0.0042,0.4181 ± 0.0045
gini,10,5,2,log2,200,0.4008 ± 0.0093,0.4154 ± 0.0083,0.4149 ± 0.0112
gini,10,5,2,sqrt,200,0.4008 ± 0.0093,0.4154 ± 0.0083,0.4149 ± 0.0112
entropy,10,5,1,log2,200,0.4004 ± 0.0174,0.4171 ± 0.0170,0.4176 ± 0.0179
entropy,10,5,1,sqrt,200,0.4004 ± 0.0174,0.4171 ± 0.0170,0.4176 ± 0.0179
